In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q s3fs xarray netCDF4 h5py scikit-image piq lpips tensorboard tqdm matplotlib pandas

# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 1. Unzip the codebase
!unzip -q /content/ISRO-Hackathon-ka-kaam-main.zip -d /content/
!mkdir -p /content/project
!cp -r /content/ISRO-Hackathon-ka-kaam-main/* /content/project/

# Move into the project directory
%cd /content/project

# 2. Pull the data from your Drive vault
!cp /content/drive/MyDrive/ISRO_Hackathon/hurricane_dataset.h5 .

# 3. Unpack and create triplets
!python scripts/unpack_h5.py
!python scripts/create_triplets.py --frames data/processed/frames/ --output data/processed/ --stride 2

In [ ]:
import torch
import sys
sys.path.insert(0, '/content/project')

from models.pipeline import SatelliteInterpolator
from models.losses import TwoModelLoss
from dataset import SyntheticSatelliteDataset

device = torch.device('cuda')

# Create model
model = SatelliteInterpolator().to(device)
model.summary()

# Test forward pass
ds = SyntheticSatelliteDataset(num_samples=10, image_size=256)
f0, gt, f1 = ds[0]
f0 = f0.unsqueeze(0).to(device)
gt = gt.unsqueeze(0).to(device)
f1 = f1.unsqueeze(0).to(device)

# Forward pass
with torch.no_grad():
    out = model(f0, f1, t=0.5, refine=True)

print(f"\nInput shape:    {f0.shape}")
print(f"Coarse shape:   {out['coarse'].shape}")
print(f"Refined shape:  {out['refined'].shape}")
print(f"Flow shape:     {out['flow'].shape}")
print(f"Residual range: [{out['residual'].min():.4f}, {out['residual'].max():.4f}]")

print("\n✅ Sanity test passed!")

In [ ]:
# Stage 1: Train Optical Flow
!python train_stage1.py \
  --data_dir data/processed \
  --epochs 100 \
  --batch_size 8 \
  --crop_size 256 \
  --save_dir /content/drive/MyDrive/ISRO_Hackathon/checkpoints/stage1

In [ ]:
# Stage 2: Train Refinement Network
!python train_stage2.py \
  --data_dir data/processed \
  --epochs 50 \
  --batch_size 8 \
  --save_dir /content/drive/MyDrive/ISRO_Hackathon/checkpoints/stage2

In [ ]:
# Stage 3: End-to-End Finetuning (50 Epoch Run)
!python train_e2e.py \
  --data_dir data/processed \
  --resume /content/drive/MyDrive/ISRO_Hackathon/checkpoints/stage2/refinement_model_best.pth \
  --epochs 50 \
  --batch_size 4 \
  --save_dir /content/drive/MyDrive/ISRO_Hackathon/checkpoints/e2e_50_epochs

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
from models.pipeline import SatelliteInterpolator
from dataset import SyntheticSatelliteDataset

device = torch.device('cuda')
model = SatelliteInterpolator().to(device)

# Load best checkpoint from the 50-epoch run
ckpt = torch.load('/content/drive/MyDrive/ISRO_Hackathon/checkpoints/e2e_50_epochs/e2e_model_best.pth', map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()

# Get a test sample
ds = SyntheticSatelliteDataset(100, 256)
f0, gt, f1 = ds[0]
f0 = f0.unsqueeze(0).to(device)
gt_tensor = gt.unsqueeze(0).to(device)
f1 = f1.unsqueeze(0).to(device)

with torch.no_grad():
    out = model(f0, f1, t=0.5, refine=True)

# Visualize
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
images = [
    ('Frame t₀', f0),
    ('Ground Truth', gt_tensor),
    ('Frame t₁', f1),
    ('Coarse (Model 1)', out['coarse']),
    ('Refined (Model 1+2)', out['refined']),
    ('Residual (×10)', out['residual'] * 10 + 0.5),
    ('Blend Mask', out['mask']),
    ('|Error| (×5)', (torch.abs(out['refined'] - gt_tensor) * 5)),
]
for ax, (title, img) in zip(axes.flat, images):
    ax.imshow(img.squeeze().cpu().numpy(), cmap='inferno', vmin=0, vmax=1)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')
plt.suptitle('2-Model Satellite Frame Interpolation Results', fontsize=16, fontweight='bold')
plt.tight_layout()

os.makedirs('output', exist_ok=True)
plt.savefig('output/visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to output/visualization.png")

In [ ]:
# Load the TensorBoard extension for Colab
%load_ext tensorboard

# Launch it directly inside the notebook targeting the project runs folder
%tensorboard --logdir /content/project/runs/